# Module 4: Building Decision Tree Prompts

## Learning Objectives

By completing this notebook, you will:
1. Build prompts that produce conditional tree responses instead of flat verdicts
2. Compare the information density of flat answers vs. structured tree answers
3. Implement a three-stage pipeline: extract conditions → identify branches → answer per branch
4. Apply conditional tree prompting to three real domains: business decisions, code architecture, and compliance

## Prerequisites
- Anthropic API key set as `ANTHROPIC_API_KEY` environment variable
- Modules 1–3 completed (Bayesian Frame, Switch Variables, Condition Stack)

## Estimated Time: 15 minutes

## Setup

We use the Anthropic Python SDK. Every call follows the same pattern: `client.messages.create()` with a model, max tokens, and message list.

In [ ]:
import anthropic
import json
import textwrap

client = anthropic.Anthropic()
MODEL = "claude-opus-4-5"


def ask(prompt: str, max_tokens: int = 1200) -> str:
    """Send a single-turn prompt and return the text response."""
    message = client.messages.create(
        model=MODEL,
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
    )
    return message.content[0].text


def display(label: str, text: str, width: int = 80) -> None:
    """Print a labelled, word-wrapped block for easy reading."""
    print(f"{'=' * width}")
    print(f"  {label}")
    print(f"{'=' * width}")
    # Preserve existing newlines; wrap only long lines
    for paragraph in text.split("\n"):
        if paragraph.strip() == "":
            print()
        else:
            print(textwrap.fill(paragraph, width=width))
    print()


print("Setup complete. Client ready.")

---

## Part 1: The Three-Stage Pipeline

A conditional tree prompt pipeline has three stages:

1. **Extract conditions** — ask what variables would change the answer
2. **Identify branches** — for each condition combination, name the branch
3. **Answer per branch** — produce a specific answer for each branch

We can run these as separate prompts (cleaner structure) or combine them in one prompt (faster for exploration). We'll implement both.

### The Pipeline Functions

In [ ]:
def extract_conditions(question: str) -> str:
    """
    Stage 1: Ask the model to identify which conditions would change
    its answer — without answering the question yet.
    
    Returns the raw text of the conditions list.
    """
    prompt = f"""You are analyzing a question to identify its hidden conditional structure.

For the question below, list the 4-6 conditions that would most change the correct
answer. For each condition, state:
- The condition (what could be true or false)
- What changes if it is true vs. false

Do NOT answer the question yet. Only identify the conditions.

Question: {question}"""
    return ask(prompt)


def identify_branches(question: str, conditions: str) -> str:
    """
    Stage 2: Given the conditions, identify the major decision branches
    (the meaningful combinations of conditions that lead to different answers).
    
    Returns the branch structure.
    """
    prompt = f"""Given this question and the conditions that would change its answer,
identify the 3-5 major branches — the meaningful combinations of conditions
that lead to clearly different answers.

Name each branch and describe the condition profile that puts someone on that branch.

Question: {question}

Conditions identified:
{conditions}

List the major branches:"""
    return ask(prompt)


def answer_per_branch(question: str, branches: str) -> str:
    """
    Stage 3: Answer the question specifically for each branch.
    
    Returns a structured response with one answer per branch.
    """
    prompt = f"""Answer this question specifically for each branch listed below.
For each branch:
1. State the branch name and its condition profile
2. Give the specific recommendation for that branch
3. Give one concrete first action someone on that branch should take

Question: {question}

Branches:
{branches}"""
    return ask(prompt, max_tokens=1600)


def flat_answer(question: str) -> str:
    """
    Baseline: ask the question directly with no conditional framing.
    Used for comparison with the pipeline output.
    """
    return ask(question, max_tokens=400)


print("Pipeline functions defined.")

---

## Example 1: Business Decision

**Question:** "Should I raise venture capital or bootstrap my startup?"

This is a canonical question with deep conditional structure. The right answer depends on at least five or six variables — but most AI responses collapse it into a verdict about the "typical" case.

We'll first get the flat answer, then run the pipeline, and compare.

In [ ]:
question_biz = "Should I raise venture capital or bootstrap my startup?"

# Step 0: Get the flat (baseline) answer
flat = flat_answer(question_biz)
display("FLAT ANSWER (baseline)", flat)

In [ ]:
# Stage 1: Extract conditions
conditions_biz = extract_conditions(question_biz)
display("STAGE 1 — CONDITIONS", conditions_biz)

In [ ]:
# Stage 2: Identify branches
branches_biz = identify_branches(question_biz, conditions_biz)
display("STAGE 2 — BRANCHES", branches_biz)

In [ ]:
# Stage 3: Answer per branch
tree_answer_biz = answer_per_branch(question_biz, branches_biz)
display("STAGE 3 — ANSWER PER BRANCH", tree_answer_biz)

### Reflection — Example 1

Notice the difference between the flat answer and the pipeline output:

- **Flat answer:** Likely contains hedging phrases like "it depends," "both have advantages," or a slight lean toward one option framed as the "typical" recommendation.
- **Pipeline output:** Identifies specific conditions, names distinct branches, and gives actionable recommendations for each.

**Exercise 1.1:** Read the conditions from Stage 1. Which three conditions do you think have the highest decision leverage (i.e., would most change your specific situation)? Write them here:

1. _______________
2. _______________
3. _______________

Then, based on the branches in Stage 2, identify which branch you (or a hypothetical founder you know) would be on.

---

## Example 2: Code Architecture Choice

**Question:** "Should I use a SQL or NoSQL database for my application?"

Architecture decisions have strong conditional structure based on data model, scale, team experience, and query patterns. The training prior strongly favors SQL for most contexts — but "most contexts" hides the cases where NoSQL is clearly superior.

In [ ]:
question_arch = "Should I use a SQL or NoSQL database for my application?"

# Run all three stages
flat_arch = flat_answer(question_arch)
conditions_arch = extract_conditions(question_arch)
branches_arch = identify_branches(question_arch, conditions_arch)
tree_answer_arch = answer_per_branch(question_arch, branches_arch)

display("FLAT ANSWER (baseline)", flat_arch)
display("STAGE 1 — CONDITIONS", conditions_arch)
display("STAGE 2 — BRANCHES", branches_arch)
display("STAGE 3 — ANSWER PER BRANCH", tree_answer_arch)

### Reflection — Example 2

Architecture decisions are particularly prone to the "average answer" failure mode because:

1. The training prior is strong (SQL dominates most use cases in training data)
2. The stakes of getting it wrong are high (hard to migrate later)
3. The conditions that flip the answer are specific and easy to know upfront

**Exercise 2.1:** The pipeline produces a set of branches. For each branch, identify one real application or product type that would be on that branch. This helps calibrate whether the branches are accurate and complete.

---

## Example 3: Compliance Question

**Question:** "Does my startup need to comply with SOC 2?"

Compliance questions look like yes/no questions. They are almost always decision trees with jurisdiction, industry, customer type, and data type as branch conditions. A flat "yes" or "no" can be both technically correct and dangerously misleading.

In [ ]:
question_compliance = "Does my startup need to comply with SOC 2?"

# Flat answer first — this is where compliance questions are most dangerous
flat_compliance = flat_answer(question_compliance)
display("FLAT ANSWER (baseline)", flat_compliance)

In [ ]:
# Run the full pipeline
conditions_compliance = extract_conditions(question_compliance)
branches_compliance = identify_branches(question_compliance, conditions_compliance)
tree_answer_compliance = answer_per_branch(question_compliance, branches_compliance)

display("STAGE 1 — CONDITIONS", conditions_compliance)
display("STAGE 2 — BRANCHES", branches_compliance)
display("STAGE 3 — ANSWER PER BRANCH", tree_answer_compliance)

### Reflection — Example 3

Compliance questions are one of the highest-stakes categories for conditional tree prompting. Compare:

- **Flat answer:** Probably says something like "SOC 2 is not legally required but is often expected by enterprise customers..."
- **Pipeline output:** Shows the actual decision tree — enterprise SaaS selling to security-conscious buyers is a different branch from a small B2C app

**Exercise 3.1:** The flat compliance answer often contains the phrase "it depends on your situation." Count how many times this or similar phrases appear in the flat answer. Then count how many times the pipeline output gives you a concrete answer for a specific branch. The ratio shows the information gain from conditional tree prompting.

---

## Part 2: Single-Prompt Conditional Tree

The three-stage pipeline is useful for exploration. For production use, you often want a single prompt that produces the full conditional tree in one call.

Here is the combined prompt pattern:

In [ ]:
def conditional_tree_prompt(question: str, domain_context: str = "") -> str:
    """
    Single-shot conditional tree prompt.
    
    Produces a complete decision tree response in one API call.
    Useful for production pipelines where latency matters.
    
    Parameters
    ----------
    question : str
        The question to answer as a conditional tree.
    domain_context : str
        Optional context about the domain (e.g., "B2B SaaS startup").
        If provided, the tree focuses on branches relevant to this context.
    
    Returns
    -------
    str
        A structured conditional tree response.
    """
    context_line = f"\nContext about the asker: {domain_context}" if domain_context else ""
    
    prompt = f"""Answer the following question as a decision tree, not a single verdict.{context_line}

Structure your response as follows:

HIDDEN CONDITIONS
List the 3-5 conditions that would change the correct answer.
For each: state the condition and what changes when it's true vs. false.

MAJOR BRANCHES
Name 3-5 distinct branches based on the conditions above.
For each branch, describe the condition profile.

ANSWER FOR EACH BRANCH
For each branch: specific recommendation + first action to take.

QUESTIONS TO ASK FIRST
If the person hasn't specified their conditions, list the 2-3 questions
they should answer before acting.

Question: {question}"""
    
    return ask(prompt, max_tokens=1400)


print("Single-shot prompt function defined.")

In [ ]:
# Test the single-shot prompt with the business question
single_shot_result = conditional_tree_prompt(
    "Should I raise venture capital or bootstrap my startup?",
    domain_context="early-stage founder, technical background, B2B SaaS product"
)
display("SINGLE-SHOT CONDITIONAL TREE", single_shot_result)

---

## Part 3: Comparing Quality Metrics

We can measure the information quality difference between flat answers and conditional tree answers using simple heuristics.

These metrics aren't definitive evaluations — they're diagnostic signals.

In [ ]:
def count_hedges(text: str) -> int:
    """
    Count how many vague hedging phrases appear in a response.
    
    High hedge count = model is averaging over conditions instead of
    surfacing them. Low hedge count in a conditional response = good.
    """
    hedge_phrases = [
        "it depends",
        "generally speaking",
        "in most cases",
        "typically",
        "usually",
        "often",
        "for most",
        "many situations",
        "varies widely",
        "context matters",
    ]
    text_lower = text.lower()
    return sum(phrase in text_lower for phrase in hedge_phrases)


def count_actionable_recommendations(text: str) -> int:
    """
    Count phrases that signal a specific, actionable recommendation.
    
    High count in a conditional response indicates concrete per-branch advice.
    """
    action_phrases = [
        "you should",
        "choose",
        "use",
        "start with",
        "first step",
        "recommend",
        "go with",
        "consider",
        "your next action",
    ]
    text_lower = text.lower()
    return sum(phrase in text_lower for phrase in action_phrases)


def quality_comparison(question: str, flat: str, tree: str) -> None:
    """
    Print a side-by-side comparison of quality metrics for flat vs. tree answers.
    """
    print(f"Question: {question[:70]}..." if len(question) > 70 else f"Question: {question}")
    print()
    print(f"{'Metric':<35} {'Flat Answer':>15} {'Tree Answer':>15}")
    print("-" * 65)
    
    flat_hedges = count_hedges(flat)
    tree_hedges = count_hedges(tree)
    flat_actions = count_actionable_recommendations(flat)
    tree_actions = count_actionable_recommendations(tree)
    
    # Word count as a proxy for information density
    flat_words = len(flat.split())
    tree_words = len(tree.split())
    
    print(f"{'Hedge phrases (lower = better)':<35} {flat_hedges:>15} {tree_hedges:>15}")
    print(f"{'Actionable recommendations':<35} {flat_actions:>15} {tree_actions:>15}")
    print(f"{'Word count':<35} {flat_words:>15} {tree_words:>15}")
    print(f"{'Actions per 100 words':<35} {flat_actions/flat_words*100:>14.1f} {tree_actions/tree_words*100:>14.1f}")
    print()


print("Quality metrics functions defined.")

In [ ]:
# Compare quality metrics for all three examples
# We use the single-shot tree for the comparison (it's the most representative
# of what you'd use in practice)

tree_biz = conditional_tree_prompt(question_biz)
tree_arch = conditional_tree_prompt(question_arch)
tree_comp = conditional_tree_prompt(question_compliance)

quality_comparison(question_biz, flat_biz, tree_biz)
quality_comparison(question_arch, flat_arch, tree_arch)
quality_comparison(question_compliance, flat_compliance, tree_comp)

---

## Part 4: Your Turn — Bring Your Own Question

Apply the conditional tree pipeline to a question from your own work or domain. Choose a question where you've received a flat AI answer that felt incomplete or generic.

**Good candidates:**
- Strategic decisions: "Should I...?"
- Architecture choices: "What's the best way to...?"
- Compliance or legal questions: "Do I need to...?"
- Hiring or team decisions: "How should I...?"

**Poor candidates:**
- Factual lookups: "What is...?"
- Syntax questions: "How do I write...?"
- Simple math or conversions

In [ ]:
# Replace this with your own question
your_question = "What is the best way to structure equity compensation for early employees?"

# Optional: add context about your situation
your_context = "early-stage B2B SaaS, pre-seed, 3 co-founders, hiring first engineer"

# Get the flat answer
your_flat = flat_answer(your_question)
display("YOUR QUESTION — FLAT ANSWER", your_flat)

# Get the conditional tree answer
your_tree = conditional_tree_prompt(your_question, your_context)
display("YOUR QUESTION — CONDITIONAL TREE", your_tree)

# Compare quality metrics
quality_comparison(your_question, your_flat, your_tree)

### Reflection — Your Question

After running the pipeline on your own question:

1. Which branch are you actually on? Can you identify it from the tree output?
2. Did the pipeline surface any conditions you hadn't thought about?
3. Is the answer for your specific branch meaningfully different from the flat answer?

If the pipeline didn't produce a better answer than the flat version, that's useful too. It might mean:
- The question genuinely doesn't have much conditional structure
- The conditions the model identified aren't the right ones for this domain
- You need to add more domain context to get a more accurate tree

---

## Summary

### Key Takeaways

1. **Flat prompts produce training priors.** When a question has conditional structure, a flat prompt returns the most common answer across all conditions — which is wrong for most specific situations.

2. **The three-stage pipeline surfaces hidden structure.** Extract conditions → identify branches → answer per branch. Each stage makes the model's implicit assumptions explicit.

3. **Single-shot conditional prompts work for production.** The structured single-shot prompt produces a decision tree in one API call — lower latency, still much better than flat answers.

4. **Quality is measurable.** Hedge count, actionable recommendation density, and branch coverage are practical signals for comparing prompt quality.

5. **Compliance and architecture decisions benefit most.** These domains have the highest cost-of-wrong-branch and the most predictable conditional structure.

### What's Next

Module 4, Exercise 1 (`exercises/01_convert_to_conditional.md`): Convert 5 single-answer prompts into conditional tree prompts. Practice identifying hidden uncertainty in questions from different domains.

Module 5: Apply conditional tree reasoning inside agents and multi-step workflows, where the decision tree is executed dynamically rather than described in text.

### Additional Resources
- `guides/01_conditional_trees_guide.md`: Conceptual foundation and technique reference
- `guides/02_uncertainty_prompting_guide.md`: Advanced uncertainty prompting techniques
- Anthropic docs: `client.messages.create()` parameter reference